# Drop size vs time (d²-law)

Reads `results/measurements.csv` from `droplet_size_measurement_v2.ipynb` and turns it into
size-vs-time curves, one per tracked drop (`drop_id`).

**What it does**
1. Loads the CSV and keeps only trustworthy rows. It prints exactly what was removed and why.
2. Fits each drop to the **d²-law**, `D(t)² = D₀² − K·t`, and reports the evaporation constant
   **K (µm²/s)** per drop. → `results/size_vs_time/drop_fits.csv`
3. Figure 1: a filmstrip of one drop, plus D vs t and D² vs t for a few example drops (the
   same layout as the earlier glare-separation plot).
4. Figure 2: every drop at once, aligned at its fitted disappearance time, plus the spread of K
   per video.
5. Figure 3: a QC list of drops whose fit looks off. These are the first place to look for a
   wrong `drop_id` link.

**How to use:** edit the Config cell, then Run All. Nothing here re-measures images; it only
reads the CSV (and, for the filmstrip only, the original video).

In [ ]:
# ============================== CONFIG ==============================
import os
DATE_DIR      = r"F:\Imaging\2026.10.12"
CSV_PATH      = os.path.join(DATE_DIR, "results", "measurements.csv")
VIDEO_DIR     = os.path.join(DATE_DIR, "drops")        # only used for the filmstrip
OUT_DIR       = os.path.join(DATE_DIR, "results", "size_vs_time")

VIDEOS        = None            # None = every video in the CSV, or e.g. ["evap_30", "evap_31"]
SIZE_COL      = "D_eq_um"       # circle-fit diameter (primary). Alternatives: "mean_HV_um", "V_um"

# --- which rows / drops to trust ---------------------------------------
MIN_FRAMES    = 8      # a drop needs at least this many frames to be fitted / plotted
SHARPNESS_MIN = 1.5    # rows below this are out of focus (soft, oversized shadows: the ~40-47 um
                       # single-frame "drops" in evap_29 are all < 1.25). In-focus drops sit ~2-2.5.
DROP_FIT_FAILED = True # discard rows where the circle fit failed

# --- d2-law fit ----------------------------------------------------------
FIT_SKIP_FIRST = 0     # skip this many frames at the start of each drop (entry frame can be distorted)
FIT_MIN_D_UM   = 0.0   # ignore points smaller than this in the fit (the tiniest drops are ~4 px)
FIT_OUTLIER_UM = 2.5   # a point is an outlier if its DIAMETER is more than this many um off the drop's
                       # fitted curve (2.5 um ~ 1.5 px). One fixed number for every drop and every video.
                       # Typical scatter is 0.3-0.9 um, so this only catches real dips (a flared glare
                       # spot on one frame). Set to None to fit every point.

# --- figures --------------------------------------------------------------
EXAMPLE_VIDEO   = "evap_31"   # video the example drops in Figure 1 come from
EXAMPLE_DROPS   = None        # None = the N_EXAMPLES longest drops; or a list of drop_id strings
N_EXAMPLES      = 6
FILMSTRIP_DROP  = None        # None = first example drop
FILMSTRIP_EVERY = 3           # show every Nth frame of that drop
FILMSTRIP_HALF  = 20          # half-width (px) of each square filmstrip crop
QC_K_MAD        = 3.0         # Figure 3: flag drops whose K is > this many MADs from their video's median
QC_R2_MIN       = 0.90        # Figure 3: flag drops whose d2 fit R^2 (after outlier rejection) is below this
QC_MAX_EXCLUDED = 0.20        # Figure 3: flag drops where more than this fraction of their frames were excluded from the fit

In [ ]:
# ============================ IMPORTS & STYLE ============================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cv2

os.makedirs(OUT_DIR, exist_ok=True)

# Fixed categorical order (never cycled). Colour follows the entity: a drop keeps its colour
# wherever it appears in a figure.
SERIES = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#6250d6", "#e34948"]
INK, INK2, GRID = "#1f1f1e", "#5f5e58", "#e4e3dd"

plt.rcParams.update({
    "figure.dpi": 110, "savefig.dpi": 160, "font.size": 10,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.edgecolor": INK2, "axes.labelcolor": INK, "xtick.color": INK2, "ytick.color": INK2,
    "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.8,
    "axes.titlesize": 11, "axes.titleweight": "regular", "legend.frameon": False,
    "lines.linewidth": 1.6, "lines.markersize": 4.5,
})
print("matplotlib", plt.matplotlib.__version__, "| pandas", pd.__version__)

In [ ]:
# ======================= LOAD + FILTER =======================
if not os.path.isfile(CSV_PATH):
    raise FileNotFoundError(f"measurements.csv not found: {CSV_PATH}  (run droplet_size_measurement_v2 first)")
raw = pd.read_csv(CSV_PATH)
if VIDEOS:
    raw = raw[raw["video"].isin(VIDEOS)]
print(f"{len(raw)} rows, {raw['drop_id'].nunique()} drop_ids, videos: {sorted(raw['video'].unique())}\n")

df = raw[raw[SIZE_COL].notna()].copy()
log = [("rows with a measurement", len(df))]

if DROP_FIT_FAILED and "fit_failed" in df:
    df = df[~df["fit_failed"].astype(bool)];        log.append(("  after removing fit_failed", len(df)))
n_soft = int((df["sharpness"] < SHARPNESS_MIN).sum())
df = df[df["sharpness"] >= SHARPNESS_MIN];          log.append((f"  after sharpness >= {SHARPNESS_MIN}  ({n_soft} out-of-focus rows)", len(df)))

# frames per drop are counted AFTER the row filters, so a drop is judged on its usable frames
df["n_use"] = df.groupby("drop_id")["frame"].transform("size")
df = df[df["n_use"] >= MIN_FRAMES].copy();          log.append((f"  after drops with >= {MIN_FRAMES} frames", len(df)))
df = df.sort_values(["video", "drop_id", "frame"]).reset_index(drop=True)

# time measured from the first KEPT frame of each drop
df["t_s"] = df["time_s"] - df.groupby("drop_id")["time_s"].transform("min")
df["D2"]  = df[SIZE_COL] ** 2

for what, n in log:
    print(f"{what:<60} {n:6d}")
print()
print(df.groupby("video")["drop_id"].nunique().rename("drops kept").to_frame().T.to_string())
print(f"\n{SIZE_COL}: {df[SIZE_COL].min():.1f} - {df[SIZE_COL].max():.1f} um across kept drops")

In [ ]:
# ======================= d2-LAW FIT PER DROP =======================
# D^2 = D0^2 - K t   ->   straight line in (t, D^2); K = -slope.  The slope does not depend on
# where t = 0 is put, so the arbitrary first-detection time does not bias K.
def _line(t, D, use):
    """Weighted straight-line fit of D^2 vs t. A diameter error of s um is a D^2 error of 2*D*s, so
    each point is weighted by 1/D: every frame counts equally in DIAMETER, and the large, early
    points do not dominate the fit just because they were squared."""
    (m, b), cov = np.polyfit(t[use], D[use] ** 2, 1, w=1.0 / D[use], cov="unscaled")
    resid = (D[use] ** 2 - (m * t[use] + b)) / D[use]
    cov = cov * (resid ** 2).sum() / max(use.sum() - 2, 1)
    return m, b, cov

def D_fit(t, m, b):
    return np.sqrt(np.clip(m * t + b, 0, None))

def fit_drop(g):
    """Returns (fit dict, boolean array of which rows of g were used)."""
    t, D = g["t_s"].to_numpy(), g[SIZE_COL].to_numpy()
    use = np.ones(len(g), bool)
    use[:FIT_SKIP_FIRST] = False
    use &= D >= FIT_MIN_D_UM
    if use.sum() < 4:
        return None, use
    base = use.copy()
    m, b, cov = _line(t, D, use)
    if FIT_OUTLIER_UM:
        for _ in range(2):                      # refit after removing outliers; a second pass catches
            keep = base & (np.abs(D - D_fit(t, m, b)) <= FIT_OUTLIER_UM)   # one hidden by a bigger one
            if keep.sum() < 4 or (keep == use).all():
                break
            use = keep
            m, b, cov = _line(t, D, use)
    dres = D[use] - D_fit(t[use], m, b)
    yhat = m * t[use] + b
    y = D[use] ** 2
    ss_res, ss_tot = ((y - yhat) ** 2).sum(), ((y - y.mean()) ** 2).sum()
    return ({"K_um2_per_s": -m, "K_err": float(np.sqrt(cov[0, 0])),
             "D0_fit_um": float(np.sqrt(b)) if b > 0 else np.nan,
             "t_extinct_s": b / -m if m < 0 else np.nan,   # when the fitted D^2 reaches 0
             "r2": 1 - ss_res / ss_tot if ss_tot > 0 else np.nan,
             "rms_D_um": float(np.sqrt((dres ** 2).mean())),   # scatter about the curve, in diameter
             "n_fit": int(use.sum()), "frames_excluded": int(base.sum() - use.sum())}, use)

rows = []
df["in_fit"] = False
for did, g in df.groupby("drop_id", sort=False):
    f, use = fit_drop(g)
    df.loc[g.index, "in_fit"] = use
    if f is None:
        continue
    rows.append({"drop_id": did, "video": g["video"].iat[0],
                 "first_frame": int(g["frame"].iat[0]), "last_frame": int(g["frame"].iat[-1]),
                 "n_frames": len(g), "duration_s": g["t_s"].iat[-1],
                 "D_first_um": g[SIZE_COL].iat[0], "D_last_um": g[SIZE_COL].iat[-1],
                 "median_aspect": g["aspect"].median() if "aspect" in g else np.nan, **f})
fits = pd.DataFrame(rows)
fits.to_csv(os.path.join(OUT_DIR, "drop_fits.csv"), index=False)

summ = fits.groupby("video").agg(drops=("drop_id", "size"),
                                 K_median=("K_um2_per_s", "median"),
                                 K_q25=("K_um2_per_s", lambda s: s.quantile(.25)),
                                 K_q75=("K_um2_per_s", lambda s: s.quantile(.75)),
                                 D_first_median=("D_first_um", "median"),
                                 r2_median=("r2", "median"))
print("wrote", os.path.join(OUT_DIR, "drop_fits.csv"), f"({len(fits)} drops)\n")
print(summ.round(2).to_string())
print(f"\n{int(fits['frames_excluded'].sum())} of {int(fits['n_fit'].sum() + fits['frames_excluded'].sum())} frames excluded from the fits "
      f"(diameter > {FIT_OUTLIER_UM} um off the drop's own curve); the drops themselves are all kept")
print(f"\nall videos: K = {fits['K_um2_per_s'].median():.0f} um^2/s "
      f"(IQR {fits['K_um2_per_s'].quantile(.25):.0f}-{fits['K_um2_per_s'].quantile(.75):.0f})")

In [ ]:
# ======================= FIGURE 1: example drops =======================
ex_pool = fits[fits["video"] == EXAMPLE_VIDEO]
if EXAMPLE_DROPS:
    ex_ids = list(EXAMPLE_DROPS)
else:
    ex_ids = ex_pool.sort_values(["n_frames", "first_frame"], ascending=[False, True])["drop_id"].head(N_EXAMPLES)
    ex_ids = sorted(ex_ids, key=lambda d: int(fits.set_index("drop_id").at[d, "first_frame"]))
colour = {d: SERIES[i % len(SERIES)] for i, d in enumerate(ex_ids)}   # <= 8 examples: no cycling
fit_of = fits.set_index("drop_id")

def filmstrip(drop_id):
    """Square crops of one drop straight from the video (raw pixels, nearest-neighbour)."""
    g = df[df["drop_id"] == drop_id].iloc[::FILMSTRIP_EVERY]
    path = os.path.join(VIDEO_DIR, g["video"].iat[0] + ".avi")
    if not os.path.isfile(path):
        print(f"(filmstrip skipped: video not found at {path})")
        return None, g
    cap, crops, h = cv2.VideoCapture(path), [], FILMSTRIP_HALF
    for _, r in g.iterrows():
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(r["frame"]))
        ok, im = cap.read()
        if not ok:
            continue
        im = cv2.cvtColor(im, cv2.COLOR_BGR2GRAY)
        im = cv2.copyMakeBorder(im, h, h, h, h, cv2.BORDER_REPLICATE)
        cx, cy = int(round(r["centroid_x"])) + h, int(round(r["centroid_y"])) + h
        crops.append(im[cy - h:cy + h + 1, cx - h:cx + h + 1])
    cap.release()
    return (np.hstack([np.pad(c, ((0, 0), (0, 3)), constant_values=255) for c in crops]) if crops else None), g

strip_id = FILMSTRIP_DROP or (ex_ids[0] if len(ex_ids) else None)
strip, strip_rows = filmstrip(strip_id) if strip_id else (None, None)

fig = plt.figure(figsize=(12, 6.6 if strip is not None else 4.6), layout="constrained")
gs = fig.add_gridspec(2 if strip is not None else 1, 2, height_ratios=[1, 3.2] if strip is not None else [1])
if strip is not None:
    axs = fig.add_subplot(gs[0, :])
    lo, hi = np.percentile(strip, [1, 99.5])
    axs.imshow(strip, cmap="gray", vmin=lo, vmax=hi, interpolation="nearest", aspect="equal")
    axs.set_axis_off()
    dt = strip_rows["t_s"].diff().median()
    axs.set_title(f"{strip_id}: every {FILMSTRIP_EVERY}{'rd' if FILMSTRIP_EVERY == 3 else 'th'} frame "
                  f"(~{dt:.2f} s apart), {strip_rows[SIZE_COL].iat[0]:.0f} -> {strip_rows[SIZE_COL].iat[-1]:.0f} um",
                  loc="left", color=INK2, fontsize=10)
axD = fig.add_subplot(gs[-1, 0]); axQ = fig.add_subplot(gs[-1, 1])

for d in ex_ids:
    g, c = df[df["drop_id"] == d], colour[d]
    lab = d.replace(EXAMPLE_VIDEO + "_", "")
    axD.plot(g["t_s"], g[SIZE_COL], "-o", color=c, label=lab, mec="white", mew=0.8)
    gi, go = g[g["in_fit"]], g[~g["in_fit"]]
    axQ.plot(gi["t_s"], gi["D2"], "o", color=c, mec="white", mew=0.8)
    axQ.plot(go["t_s"], go["D2"], "o", mfc="none", mec=c, mew=1.2, ms=6)   # rejected from the fit
    f = fit_of.loc[d]
    tt = np.linspace(g["t_s"].min(), g["t_s"].max(), 50)
    axQ.plot(tt, f["D0_fit_um"] ** 2 - f["K_um2_per_s"] * tt, "-", color=c, lw=1.4, alpha=0.9)

axD.set(xlabel="time since drop appeared (s)", ylabel=f"diameter, {SIZE_COL.replace('_um', '')} (µm)")
axD.set_title(f"{EXAMPLE_VIDEO}: drop diameter vs time", loc="left")
axD.legend(ncol=2, fontsize=9, title="drop", title_fontsize=9)
axQ.set(xlabel="time since drop appeared (s)", ylabel="diameter² (µm²)")
axQ.plot([], [], "o", mfc="none", mec=INK2, mew=1.2, ms=6, label="excluded from fit (outlier)")
axQ.legend(fontsize=9, loc="upper right")
k_ex = fit_of.loc[ex_ids, "K_um2_per_s"]
axQ.set_title(f"d²-law check: straight lines, K = {k_ex.median():.0f} µm²/s (median of these)", loc="left")
for ax in (axD, axQ):
    ax.set_xlim(left=-0.03); ax.set_ylim(bottom=0)
fig.savefig(os.path.join(OUT_DIR, f"fig1_examples_{EXAMPLE_VIDEO}.png"), bbox_inches="tight")
plt.show()

In [ ]:
# ======================= FIGURE 1b: D vs time, every drop, one panel per video =======================
# Same data as Figure 1, for every kept drop. Points left out of the d2 fit are drawn as hollow
# circles, so you can see exactly what was excluded and judge the threshold yourself.
vids = sorted(fits["video"].unique())
fig, axes = plt.subplots(1, len(vids), figsize=(4.3 * len(vids), 4.2), sharex=True, sharey=True,
                         layout="constrained", squeeze=False)
for ax, v in zip(axes[0], vids):
    dv = df[df["video"] == v]
    for d, g in dv.groupby("drop_id"):
        ax.plot(g["t_s"], g[SIZE_COL], "-", color=SERIES[vids.index(v)], lw=0.8, alpha=0.45)
        out = g[~g["in_fit"]]
        ax.plot(out["t_s"], out[SIZE_COL], "o", mfc="none", mec=INK, mew=1.0, ms=5)
    s = fits[fits["video"] == v]
    ax.set_title(f"{v}: {len(s)} drops, K = {s['K_um2_per_s'].median():.0f} µm²/s\n"
                 f"scatter about fit {s['rms_D_um'].median():.2f} µm (median), "
                 f"{int(s['frames_excluded'].sum())} frames excluded", loc="left", fontsize=10)
    ax.set_xlabel("time since drop appeared (s)")
axes[0][0].set_ylabel(f"diameter, {SIZE_COL.replace('_um', '')} (µm)")
axes[0][0].plot([], [], "o", mfc="none", mec=INK, mew=1.0, ms=5, label=f"excluded (> {FIT_OUTLIER_UM} µm off fit)")
axes[0][0].legend(fontsize=9, loc="lower left")
for ax in axes[0]:
    ax.set_ylim(bottom=0); ax.set_xlim(left=-0.03)
fig.savefig(os.path.join(OUT_DIR, "fig1b_D_vs_t_per_video.png"), bbox_inches="tight")
plt.show()

In [ ]:
# ======================= FIGURE 2: all drops =======================
# Left: every drop's D^2, with time shifted so each drop's fitted disappearance is t = 0. If all
# drops evaporate at the same K they collapse onto one line, whatever size they were first seen at.
# Right: the per-drop K, one dot per drop, per video, with the median marked.
vcol = {v: SERIES[i % len(SERIES)] for i, v in enumerate(vids)}

fig, (axA, axK) = plt.subplots(1, 2, figsize=(12, 4.6), layout="constrained",
                               gridspec_kw={"width_ratios": [1.6, 1]})
for d, g in df.groupby("drop_id"):
    if d not in fit_of.index or not np.isfinite(fit_of.at[d, "t_extinct_s"]):
        continue
    v = g["video"].iat[0]
    axA.plot(g["t_s"] - fit_of.at[d, "t_extinct_s"], g["D2"], "-", color=vcol[v], lw=0.9, alpha=0.45)
kall = fits["K_um2_per_s"].median()
tline = np.linspace(-fits["t_extinct_s"].max(), 0, 50)
axA.plot(tline, -kall * tline, "--", color=INK, lw=1.3, label=f"median K = {kall:.0f} µm²/s")
for v in vids:
    axA.plot([], [], "-", color=vcol[v], lw=2, label=f"{v} ({(fits['video'] == v).sum()} drops)")
axA.set(xlabel="time relative to fitted disappearance (s)", ylabel="diameter² (µm²)")
axA.set_title("All tracked drops, aligned at disappearance", loc="left")
axA.set_ylim(bottom=0); axA.legend(fontsize=9)

rng = np.random.default_rng(0)
for i, v in enumerate(vids):
    k = fits.loc[fits["video"] == v, "K_um2_per_s"].to_numpy()
    axK.scatter(i + rng.uniform(-0.18, 0.18, len(k)), k, s=22, color=vcol[v], ec="white", lw=0.6, zorder=3)
    med = np.median(k)
    axK.plot([i - 0.3, i + 0.3], [med, med], color=INK, lw=2, zorder=4)
    axK.annotate(f"{med:.0f}", (i + 0.32, med), va="center", fontsize=9, color=INK)
axK.set_xticks(range(len(vids)), vids); axK.grid(axis="x", visible=False)
axK.set(ylabel="K (µm²/s)"); axK.set_title("Evaporation constant per drop", loc="left")
axK.set_xlim(-0.6, len(vids) - 0.2)
fig.savefig(os.path.join(OUT_DIR, "fig2_all_drops.png"), bbox_inches="tight")
plt.show()

In [ ]:
# ======================= FIGURE 3: QC - drops worth a second look =======================
# FLAGGING REMOVES NOTHING. Every drop below is still in drop_fits.csv and in the K medians.
# A flag only means "open this drop's filmstrip (results/qc_tracks/<video>/<drop_id>.png) and check
# it". A wrong drop_id link (two drops stitched together, or one drop split) typically shows up as
# one of these. Three independent reasons, a drop can trip more than one:
#   K_outlier            K is more than QC_K_MAD robust deviations (MAD) from THAT VIDEO's median K
#   poor_fit             R^2 of the d2 fit (on the frames kept) is below QC_R2_MIN
#   many_frames_excluded more than QC_MAX_EXCLUDED of the drop's frames were left out of its fit
vstats = fits.groupby("video")["K_um2_per_s"].agg(
    K_med="median", K_mad=lambda s: 1.4826 * (s - s.median()).abs().median())
fits = fits.drop(columns=[c for c in ("K_med", "K_mad") if c in fits]).merge(vstats, on="video")
fits["frames_total"] = fits["n_fit"] + fits["frames_excluded"]
fits["K_outlier"] = (fits["K_um2_per_s"] - fits["K_med"]).abs() > QC_K_MAD * fits["K_mad"]
fits["poor_fit"] = fits["r2"] < QC_R2_MIN
fits["many_frames_excluded"] = fits["frames_excluded"] > QC_MAX_EXCLUDED * fits["frames_total"]

def why(r):
    out = []
    if r["K_outlier"]:
        out.append(f"K {r['K_um2_per_s']:.0f} vs {r['video']} median {r['K_med']:.0f} "
                   f"(allowed +-{QC_K_MAD * r['K_mad']:.0f})")
    if r["poor_fit"]:
        out.append(f"R2 {r['r2']:.2f} < {QC_R2_MIN}")
    if r["many_frames_excluded"]:
        out.append(f"{int(r['frames_excluded'])} of {int(r['frames_total'])} frames excluded "
                   f"({r['frames_excluded'] / r['frames_total']:.0%} > {QC_MAX_EXCLUDED:.0%})")
    return "; ".join(out)

fits["why_flagged"] = fits.apply(why, axis=1)
flag = fits[fits["why_flagged"] != ""]

fig, ax = plt.subplots(figsize=(7.5, 4.2), layout="constrained")
for v in vids:
    s = fits[fits["video"] == v]
    ax.scatter(s["D_first_um"], s["K_um2_per_s"], s=24, color=vcol[v], ec="white", lw=0.6, label=v, zorder=3)
ax.scatter(flag["D_first_um"], flag["K_um2_per_s"], s=90, facecolor="none", ec=INK, lw=1.2,
           label="flagged drop (kept; see table)", zorder=4)
ax.set(xlabel="diameter at first kept frame (µm)", ylabel="K (µm²/s)")
ax.set_title("One dot per drop. K should not depend on how big the drop was when first seen", loc="left")
ax.legend(fontsize=9)
fig.savefig(os.path.join(OUT_DIR, "fig3_qc.png"), bbox_inches="tight")
plt.show()

print(f"{len(flag)} of {len(fits)} drops flagged for review (one row = one drop = one circle above).")
print("Flagged drops are NOT removed. 'frames_excluded' = single frames left out of that drop's fit.\n")
if len(flag):
    with pd.option_context("display.width", 250, "display.max_colwidth", 120):
        print(flag[["drop_id", "frames_total", "frames_excluded", "D_first_um", "D_last_um",
                    "K_um2_per_s", "r2", "why_flagged"]].round(2).to_string(index=False))
fits.drop(columns=["K_med", "K_mad"]).to_csv(os.path.join(OUT_DIR, "drop_fits.csv"), index=False)

## Notes
- **Outliers** use one fixed rule: a frame is left out of the fit if its diameter is more than
  `FIT_OUTLIER_UM` (2.5 µm ≈ 1.5 px) off that drop's fitted curve. The fit is weighted by 1/D, so
  each frame counts equally in diameter. The excluded frames are almost all *dips*, frames where
  one glare spot flares up and eats into the dark disk. They are measurement artifacts, not real
  shrink-and-regrow. They will largely disappear with the laser off. They stay in the CSV, are drawn hollow in Figures 1 and 1b, and
  `frames_excluded` counts them per drop.
- **K** is reported per drop; `drop_fits.csv` also has `K_err`, the standard error from the fit.
  The spread *between* drops is the more honest uncertainty.
- The d²-law (`D²` linear in `t`) is the classic result for diffusion-limited evaporation of a
  drop in still gas. Straight lines in the right-hand panel of Figure 1 are the check that it
  holds here.
- `SIZE_COL = "D_eq_um"` is the circle fit on glare-free edge points. While the laser is on,
  `H_um` reads a little small (see `CONTEXT.md` §3). Re-run with `"V_um"` to see how much K
  depends on that choice.
- `t = 0` is the first frame the drop was *kept*, not when it was made. Drops arrive from the
  dispenser and are first seen at somewhat different sizes. Figure 2 aligns them at the fitted
  disappearance time instead.